> **Public portfolio copy:** Saved execution outputs have been removed. The notebook source code and explanations are retained. Running the notebook requires a separately authorised DICOM input file in the local `Data/` folder. No Stanford COCA image or dataset file is included in this repository.

# DICOM Image Visualization and Windowing

**Project 2: DICOM Extension – Medical Image Viewer and Preprocessing**

## Overview

This notebook explores the visualization of CT images stored in DICOM format, with particular emphasis on Hounsfield Unit (HU) values and CT windowing.

## Objectives

- Load and visualize a CT image from a DICOM file.
- Convert raw pixel values into Hounsfield Units.
- Understand CT windowing using Window Centre (WC) and Window Width (WW).
- Apply common CT window presets.
- Develop an interactive windowing viewer using Jupyter widgets.
- Compare original HU data with windowed image data.

**Dataset:** One CT DICOM image.

## 1. Environment Setup and DICOM Loading

In [ ]:
# IMPORT LIBRARIES FOR DICOM IMAGE VISUALIZATION

import numpy as np
import matplotlib.pyplot as plt
import pydicom

# Configure Matplotlib display
plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.cmap"] = "gray"

print("Libraries imported successfully.")

In [ ]:
# DEFINE THE PROJECT AND DATA FOLDER PATHS

from pathlib import Path

# Identify the project directory
project_path = Path.cwd().parent

# Identify the Data folder
data_path = project_path / "Data"

print("Project directory:", project_path)
print("Data directory:", data_path)

In [ ]:
# LOAD THE DICOM IMAGE FOR VISUALIZATION

# Replace this path with the same DICOM file path used in Notebook 01
dicom_file = data_path / "IM-0001-0001.dcm"

# Read the DICOM file
dcm = pydicom.dcmread(dicom_file)

# Extract the pixel array
pixel_array = dcm.pixel_array

# Convert raw pixel values to Hounsfield Units (HU)
ct_image = pixel_array * float(dcm.RescaleSlope) + float(dcm.RescaleIntercept)

print("DICOM image loaded successfully.")
print("Image dimensions:", ct_image.shape)
print("Data type:", ct_image.dtype)

In [ ]:
# VERIFY HU CONVERSION

print("Rescale Slope:", dcm.RescaleSlope)
print("Rescale Intercept:", dcm.RescaleIntercept)

print("\nRaw Pixel Values:")
print("Minimum:", pixel_array.min())
print("Maximum:", pixel_array.max())

print("\nConverted Hounsfield Units:")
print("Minimum HU:", ct_image.min())
print("Maximum HU:", ct_image.max())

print("\nSample Pixel Comparison:")

print("Raw pixel value:", pixel_array[256, 256])
print("Converted HU value:", ct_image[256, 256])

## 2. Initial CT Image Visualization

In [ ]:
# DISPLAY THE CT IMAGE

plt.figure(figsize=(8, 8))

plt.imshow(ct_image, cmap="gray")
plt.title("CT Image - Original HU Values")
plt.axis("off")

plt.show()

## 3. CT Windowing and Visualization

### 3.1 CT Windowing Function

In [ ]:
# DEFINE A CT WINDOWING FUNCTION

def apply_window(image, window_center, window_width):
    
    window_min = window_center - (window_width / 2)
    window_max = window_center + (window_width / 2)
    
    windowed_image = np.clip(image, window_min, window_max)
    
    windowed_image = (
        (windowed_image - window_min) /
        (window_max - window_min)
    )
    
    return windowed_image

In [ ]:
# APPLY A SOFT TISSUE WINDOW

window_center = 40
window_width = 400

windowed_image = apply_window(
    ct_image,
    window_center,
    window_width
)

plt.figure(figsize=(8, 8))

plt.imshow(windowed_image, cmap="gray")
plt.title(
    f"Soft Tissue Window (WC={window_center}, WW={window_width})"
)

plt.axis("off")

plt.show()

### 3.2 CT Window Presets

CT images can be displayed using different window settings depending on the anatomical structures being evaluated.

This section demonstrates commonly used CT window presets:

- Soft Tissue Window
- Lung Window
- Bone Window

Each window emphasizes different attenuation ranges and therefore different anatomical structures.

In [ ]:
# DEFINE COMMON CT WINDOW PRESETS

window_presets = {
    "Soft Tissue": {
        "center": 40,
        "width": 400
    },
    "Lung": {
        "center": -600,
        "width": 1500
    },
    "Bone": {
        "center": 500,
        "width": 2000
    }
}

print("CT Window Presets:")
for name, settings in window_presets.items():
    print(f"{name}: WC={settings['center']}, WW={settings['width']}")

### 3.3 Alternative Implementation of the CT Windowing Function

This cell presents an alternative implementation of the CT windowing function.

It performs the same fundamental operations as the earlier implementation:

- Calculation of the lower and upper HU limits.
- Clipping of image intensities outside the selected window.
- Normalization of the resulting values to a range between 0 and 1.

The alternative implementation provides an opportunity to compare different approaches to the same image-processing operation.

In [ ]:
# CREATE A REUSABLE CT WINDOWING FUNCTION

def apply_window(image, window_center, window_width):
    """
    Apply CT windowing to an image using Window Center and Window Width.
    
    This function:
    1. Calculates the lower and upper HU limits
    2. Clips values outside the window
    3. Normalizes the remaining values for display
    """
    
    # Calculate the lower and upper limits of the window
    lower_limit = window_center - window_width / 2
    upper_limit = window_center + window_width / 2
    
    # Clip image values to the selected window
    windowed_image = np.clip(image, lower_limit, upper_limit)
    
    # Normalize values to a range between 0 and 1
    windowed_image = (
        windowed_image - lower_limit
    ) / window_width
    
    return windowed_image


print("CT windowing function created successfully.")

In [ ]:
# APPLY AND DISPLAY THE THREE CT WINDOW PRESETS

# Create a figure containing three side-by-side images
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Loop through each window preset
for ax, (name, settings) in zip(axes, window_presets.items()):

    # Extract the window center and width
    window_center = settings["center"]
    window_width = settings["width"]

    # Apply the windowing function
    windowed_image = apply_window(
        ct_image,
        window_center,
        window_width
    )

    # Display the windowed CT image
    ax.imshow(windowed_image, cmap="gray")

    # Add the window information to the title
    ax.set_title(
        f"{name}\nWC={window_center}, WW={window_width}"
    )

    # Remove the axes
    ax.axis("off")

# Add an overall title
plt.suptitle("Comparison of CT Window Presets", fontsize=16)

# Adjust spacing between images
plt.tight_layout()

# Display the figure
plt.show()

## 4. Interactive CT Windowing

This section introduces interactive controls for adjusting CT window settings.

Jupyter widgets allow users to modify Window Centre (WC) and Window Width (WW) without manually changing the underlying Python code.

In [ ]:
# IMPORT IPYWIDGETS FOR INTERACTIVE CT WINDOWING

import ipywidgets as widgets
from IPython.display import display

print("Interactive visualization tools imported successfully.")

### 4.1 Basic Interactive Windowing Controls

This implementation introduces sliders for adjusting Window Centre and Window Width.

It provides the initial interactive controls for exploring different CT window settings.

In [ ]:
# CREATE INTERACTIVE CT WINDOWING CONTROLS

# Create a slider for Window Center (WC)
window_center_slider = widgets.IntSlider(
    value=40,
    min=-1000,
    max=1000,
    step=10,
    description="Window Center:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

# Create a slider for Window Width (WW)
window_width_slider = widgets.IntSlider(
    value=400,
    min=1,
    max=3000,
    step=50,
    description="Window Width:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

# Display the interactive controls
display(window_center_slider)
display(window_width_slider)

### 4.2 Button-Controlled Interactive CT Viewer

This implementation extends the basic interactive controls by introducing an Apply button.

The user can adjust the Window Centre and Window Width and then apply the selected settings to update the displayed CT image.

This approach provides greater control over when image processing and visualization are performed.

In [ ]:
# CREATE A BUTTON-CONTROLLED INTERACTIVE CT WINDOWING VIEWER

import ipywidgets as widgets
from IPython.display import display, clear_output

# Create Window Center slider
window_center_slider = widgets.IntSlider(
    value=40,
    min=-1000,
    max=1000,
    step=10,
    description="Window Center:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

# Create Window Width slider
window_width_slider = widgets.IntSlider(
    value=400,
    min=1,
    max=3000,
    step=50,
    description="Window Width:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

# Create the Apply button
apply_button = widgets.Button(
    description="Apply Window Settings",
    button_style="success",
    icon="check"
)

# Create an output area for the CT image
image_output = widgets.Output()

# Define the function that updates the CT image
def update_ct_display(button):
    
    # Clear the previous image
    with image_output:
        clear_output(wait=True)
        
        # Apply the selected window settings
        windowed_image = apply_window(
            ct_image,
            window_center_slider.value,
            window_width_slider.value
        )
        
        # Display the updated CT image
        plt.figure(figsize=(8, 8))
        plt.imshow(windowed_image, cmap="gray")
        plt.title(
            f"CT Windowing\n"
            f"Window Center = {window_center_slider.value}, "
            f"Window Width = {window_width_slider.value}"
        )
        plt.axis("off")
        plt.show()

# Connect the button to the update function
apply_button.on_click(update_ct_display)

# Display the controls
display(window_center_slider)
display(window_width_slider)
display(apply_button)

# Display the image output area
display(image_output)

# Display the initial CT image
update_ct_display(None)

## 5. Comparison of Original and Windowed CT Images

This section compares the original CT image in Hounsfield Units with the image displayed using soft-tissue window settings.

The comparison demonstrates how windowing changes the displayed intensity range while retaining the original image data.

In [ ]:
# COMPARE ORIGINAL HU DATA WITH WINDOWED IMAGE DATA

# Apply soft tissue windowing
soft_tissue_image = apply_window(
    ct_image,
    window_center=40,
    window_width=400
)

# Display original CT image and windowed image side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Display original CT image in HU
axes[0].imshow(ct_image, cmap="gray")
axes[0].set_title("Original CT Image (HU)")
axes[0].axis("off")

# Display windowed image
axes[1].imshow(soft_tissue_image, cmap="gray")
axes[1].set_title("Soft Tissue Window")
axes[1].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

## 6. Verification of Original CT Data Integrity

This section examines the numerical characteristics of the original CT image and the windowed image.

It also verifies that applying windowing has not modified the original Hounsfield Unit data.

In [ ]:
# VERIFY THAT WINDOWING DOES NOT MODIFY THE ORIGINAL CT DATA

# Display statistics of the original CT image
print("Original CT Image:")
print("Minimum HU:", ct_image.min())
print("Maximum HU:", ct_image.max())
print("Mean HU:", ct_image.mean())

# Display statistics of the windowed image
print("\nWindowed Image:")
print("Minimum value:", soft_tissue_image.min())
print("Maximum value:", soft_tissue_image.max())
print("Mean value:", soft_tissue_image.mean())

# Confirm that the original CT image remains unchanged
print("\nOriginal CT image remains unchanged:")
print(np.array_equal(ct_image, pixel_array * float(dcm.RescaleSlope) + float(dcm.RescaleIntercept)))

## 7. Conclusion

This notebook demonstrated CT image visualization and windowing using DICOM data.

The following tasks were completed:

- Loading a CT DICOM image and extracting its pixel data.
- Converting raw pixel values into Hounsfield Units.
- Visualizing the original CT image.
- Implementing CT windowing using Window Centre and Window Width.
- Applying soft-tissue, lung and bone window presets.
- Comparing alternative implementations of the windowing function.
- Developing interactive windowing controls using Jupyter widgets.
- Creating a button-controlled interactive CT viewer.
- Verifying that windowing does not modify the original HU data.

### Limitations

The notebook uses a single CT DICOM image rather than a complete CT series. Multi-slice navigation, volumetric visualization and multiplanar reconstruction are outside its current scope.

The interactive viewer is a Jupyter-based prototype and provides a foundation for further development into a more comprehensive medical image viewer.